# 00 — Setup Verification & Environment Bootstrap
**Thesis:** Task-Specific NLP Pipelines vs. LLMs for Cross-Lingual Text Analysis  
**University of Trier | Computational Linguistics**

---
**Purpose of this notebook:**  
1. Mount Google Drive and set up project folder structure  
2. Install all dependencies from `requirements.txt`  
3. Verify every major library loads correctly  
4. Confirm GPU availability and memory  
5. Run a quick end-to-end smoke test (German text → translation → NER → embedding)  
6. Print a reproducibility report  

> **Run this notebook once at the start of every new Colab session** to confirm the environment is healthy.
> Expected runtime: ~8–12 minutes (first run with downloads), ~2 minutes (cached runs).

## 1. GPU & Runtime Check

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                          '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f'✅ GPU available: {gpu_info}')
else:
    print('⚠️  No GPU detected — CPU only. LLM inference will be slow.')
    print('   Tip: Runtime → Change runtime type → T4 GPU')

# Check RAM
import psutil
ram = psutil.virtual_memory()
print(f'✅ RAM: {ram.total / 1e9:.1f} GB total, {ram.available / 1e9:.1f} GB free')

# Check Python version
print(f'✅ Python {sys.version}')


## 2. Mount Google Drive

In [ ]:
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/thesis/'
    print(f'✅ Drive mounted. Project root: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = './'
    print(f'ℹ️  Not in Colab. Using local path: {os.path.abspath(PROJECT_ROOT)}')

# Create directory structure
dirs = [
    'data/raw', 'data/processed', 'data/processed/cache', 'data/samples',
    'notebooks', 'src/pipeline', 'src/llm', 'src/evaluation', 'src/utils',
    'outputs/ner', 'outputs/topics', 'outputs/frames', 'outputs/figures', 'docs'
]
for d in dirs:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
print('✅ Directory structure verified.')


## 3. Install Dependencies
> This cell installs from `requirements.txt`. On first run it takes ~5–8 minutes.  
> On subsequent runs, packages are already present and this is fast.
> 
> **Important:** Restart the runtime after installation if prompted.

In [ ]:
# Install core packages
# NOTE: torch is NOT in requirements.txt — Colab's pre-installed CUDA version is used
!pip install -q \
    transformers==4.40.1 \
    tokenizers==0.19.1 \
    datasets==2.19.0 \
    huggingface-hub==0.22.2 \
    accelerate==0.29.3 \
    sentence-transformers==2.7.0 \
    safetensors==0.4.3

!pip install -q \
    spacy==3.7.4 \
    stanza==1.8.0

!pip install -q \
    bertopic==0.16.2 \
    hdbscan==0.8.33 \
    umap-learn==0.5.6 \
    gensim==4.3.2

!pip install -q \
    unbabel-comet==2.2.1 \
    bert-score==0.3.13 \
    sacrebleu==2.4.2

!pip install -q \
    easynmt==2.0.2 \
    sacremoses==0.1.1 \
    sentencepiece==0.2.0

!pip install -q \
    statsmodels==0.14.2 \
    pingouin==0.5.4 \
    krippendorff==0.6.1 \
    bitsandbytes==0.43.1

!pip install -q \
    langdetect==1.0.9 \
    chardet==5.2.0 \
    ftfy==6.2.0 \
    textstat==0.7.3 \
    python-dotenv==1.0.1 \
    pyyaml==6.0.1 \
    jsonlines==4.0.0 \
    rich==13.7.1 \
    ipywidgets==8.1.2

# CharSplit (not on PyPI)
!pip install -q git+https://github.com/dtuggener/CharSplit.git

print('\n✅ Core packages installed.')


## 4. Download spaCy German Model

In [ ]:
import subprocess
result = subprocess.run(['python', '-m', 'spacy', 'download', 'de_core_news_lg'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ de_core_news_lg downloaded.')
else:
    # May already be installed
    import spacy
    try:
        nlp = spacy.load('de_core_news_lg')
        print('✅ de_core_news_lg already available.')
    except OSError:
        print('❌ Could not install de_core_news_lg. See error:')
        print(result.stderr)


## 5. Load HuggingFace Token
> Store your token in Colab Secrets (🔑 icon in left sidebar) as `HF_TOKEN`,  
> **OR** enter it below when prompted. Never paste tokens directly in notebook cells.

In [ ]:
import os
from dotenv import load_dotenv

# Try Colab Secrets first
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN loaded from Colab Secrets.')
except Exception:
    # Fallback: .env file (local dev)
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')
    if HF_TOKEN:
        print('✅ HF_TOKEN loaded from .env file.')
    else:
        print('⚠️  No HF_TOKEN found. LLM Inference API calls will fail.')
        print('   Add it via: Colab sidebar → Secrets → Add HF_TOKEN')

# Log in to HuggingFace (required for gated models like Mistral)
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Logged in to HuggingFace.')


## 6. Import Verification

In [ ]:
from rich.console import Console
from rich.table import Table

console = Console()
results = []

def try_import(module_name, display_name, version_attr='__version__'):
    try:
        mod = __import__(module_name)
        version = getattr(mod, version_attr, 'n/a')
        results.append(('✅', display_name, version))
    except ImportError as e:
        results.append(('❌', display_name, str(e)))

try_import('torch',              'PyTorch')
try_import('transformers',       'Transformers')
try_import('sentence_transformers', 'Sentence-Transformers')
try_import('spacy',              'spaCy')
try_import('stanza',             'Stanza')
try_import('bertopic',           'BERTopic')
try_import('gensim',             'Gensim')
try_import('easynmt',            'EasyNMT')
try_import('comet',              'COMET',            '_version')
try_import('bert_score',         'BERTScore')
try_import('sacrebleu',          'SacreBLEU')
try_import('statsmodels',        'statsmodels')
try_import('pingouin',           'Pingouin')
try_import('krippendorff',       'Krippendorff')
try_import('ftfy',               'ftfy')
try_import('langdetect',         'langdetect')
try_import('textstat',           'textstat')
try_import('sklearn',            'scikit-learn')
try_import('pandas',             'Pandas')
try_import('numpy',              'NumPy')

table = Table(title='Library Import Status')
table.add_column('Status', style='bold')
table.add_column('Library')
table.add_column('Version')
for status, name, version in results:
    table.add_row(status, name, str(version))
console.print(table)

failures = [r for r in results if r[0] == '❌']
if failures:
    print(f'\n⚠️  {len(failures)} import(s) failed. Re-run the install cell.')
else:
    print('\n✅ All imports successful.')


## 7. End-to-End Smoke Test
This cell runs a short German text through the full pipeline to confirm everything connects.  
**German proficiency not required** — this is purely a technical validation.
> Test text: one sentence about sustainable building — the thesis domain.

In [ ]:
print('=== SMOKE TEST: Full Mini-Pipeline ===')
print()

TEST_DE = (
    'Das Passivhaus-Konzept revolutioniert den Wohnungsbau in Deutschland. '
    'Durch hocheffiziente Wärmedämmung können Gebäude bis zu 90 Prozent '
    'Heizenergie einsparen. Die DGNB-Zertifizierung wurde 2018 eingeführt.'
)

# ── Step 1: Encoding fix ────────────────────────────────────────────────────
import ftfy
clean_text = ftfy.fix_text(TEST_DE)
print(f'[1/6] ftfy encoding fix ... ✅')

# ── Step 2: Language detection ──────────────────────────────────────────────
from langdetect import detect
lang = detect(clean_text)
assert lang == 'de', f'Expected German, got {lang}'
print(f'[2/6] Language detected: {lang} ✅')

# ── Step 3: spaCy German NLP ────────────────────────────────────────────────
import spacy
nlp = spacy.load('de_core_news_lg')
doc = nlp(clean_text)
entities = [(ent.text, ent.label_) for ent in doc.ents]
print(f'[3/6] spaCy NER entities: {entities} ✅')

# ── Step 4: Sentence embeddings ─────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embedding = embed_model.encode(clean_text)
print(f'[4/6] Sentence embedding shape: {embedding.shape} ✅')

# ── Step 5: Machine translation ─────────────────────────────────────────────
# EasyNMT downloads model on first use (~300MB) — uses cache afterwards
print('[5/6] Loading OPUS-MT translation model (first run downloads ~300MB)...')
from easynmt import EasyNMT
translator = EasyNMT('opus-mt')
translation = translator.translate(clean_text, source_lang='de', target_lang='en')
print(f'[5/6] Translation:\n  DE: {clean_text[:80]}...\n  EN: {translation[:80]}... ✅')

# ── Step 6: Readability metrics ─────────────────────────────────────────────
import textstat
fk = textstat.flesch_reading_ease(translation)
wc = len(clean_text.split())
print(f'[6/6] Text stats — words: {wc}, Flesch (EN translation): {fk:.1f} ✅')

print()
print('=== ✅ ALL SMOKE TESTS PASSED ===')


## 8. German Compound Splitter Test

In [ ]:
try:
    from charSplit import charSplit
    test_compounds = ['Passivhaus', 'Wärmedämmung', 'Nachhaltigkeitszertifizierung', 'Lüftungsanlage']
    print('Compound splitting results:')
    for word in test_compounds:
        splits = charSplit.split_compound(word)
        print(f'  {word:35s} → {splits[0]}')
    print('\n✅ CharSplit working.')
except ImportError:
    print('⚠️  CharSplit not installed. Run: pip install git+https://github.com/dtuggener/CharSplit.git')
except Exception as e:
    print(f'⚠️  CharSplit error: {e}')


## 9. Stanza German NER Test (Second NER System)

In [ ]:
import stanza
# Downloads German models on first run (~500MB)
print('Downloading Stanza German models (first run only)...')
stanza.download('de', processors='tokenize,ner', verbose=False)
stanza_nlp = stanza.Pipeline('de', processors='tokenize,ner', verbose=False)

stanza_doc = stanza_nlp(TEST_DE)
stanza_ents = [(ent.text, ent.type) for sent in stanza_doc.sentences for ent in sent.ents]
print(f'Stanza NER entities: {stanza_ents}')
print('✅ Stanza working.')


## 10. HuggingFace Inference API Test (LLM Baseline)
> This tests the **free** HF Inference API with Mistral-7B.  
> Will fail gracefully if no token is set — that's OK for now.

In [ ]:
if HF_TOKEN:
    try:
        from huggingface_hub import InferenceClient
        client = InferenceClient(token=HF_TOKEN)
        
        # Minimal test — single short prompt
        prompt = (
            'Classify the following German text as about: '
            '[energy_efficiency / materials / certification / policy / other].\n'
            f'Text: "{TEST_DE[:150]}"\n'
            'Answer with one label only:'
        )
        response = client.text_generation(
            prompt,
            model='mistralai/Mistral-7B-Instruct-v0.3',
            max_new_tokens=20,
            temperature=0.0,
        )
        print(f'Mistral-7B response: "{response.strip()}"')
        print('✅ HF Inference API working.')
    except Exception as e:
        print(f'⚠️  Inference API test failed: {e}')
        print('   This may be a rate limit or model loading issue. Retry later.')
else:
    print('⏭️  Skipping Inference API test — no HF_TOKEN set.')


## 11. Load & Verify Config

In [ ]:
import yaml

CONFIG_PATH = os.path.join(PROJECT_ROOT, 'config.yaml')

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        config = yaml.safe_load(f)
    print(f'✅ config.yaml loaded.')
    print(f'   Random seed: {config["project"]["random_seed"]}')
    print(f'   Primary translation model: {config["translation"]["primary_model"]}')
    print(f'   Validation sample size: {config["sampling"]["total_n"]}')
    print(f'   LLM primary model: {config["llm"]["primary_model"]}')
else:
    print(f'⚠️  config.yaml not found at {CONFIG_PATH}. Copy it from the repo.')

# Set global random seed
import random, numpy as np, torch
SEED = config['project']['random_seed'] if 'config' in dir() else 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'✅ Random seed {SEED} set for random / numpy / torch.')


## 12. Reproducibility Snapshot

In [ ]:
import datetime, platform

snapshot = {
    'timestamp': datetime.datetime.now().isoformat(),
    'python_version': sys.version,
    'platform': platform.platform(),
    'torch_version': str(__import__('torch').__version__),
    'transformers_version': str(__import__('transformers').__version__),
    'spacy_version': str(__import__('spacy').__version__),
    'random_seed': SEED,
    'project_root': PROJECT_ROOT,
    'gpu_available': bool(result.returncode == 0) if 'result' in dir() else 'unknown',
}

import json, os
snapshot_path = os.path.join(PROJECT_ROOT, 'docs', 'environment_snapshot.json')
os.makedirs(os.path.dirname(snapshot_path), exist_ok=True)
with open(snapshot_path, 'w') as f:
    json.dump(snapshot, f, indent=2)

print('=== ENVIRONMENT SNAPSHOT ===')
for k, v in snapshot.items():
    print(f'  {k:25s}: {v}')
print(f'\n✅ Snapshot saved to {snapshot_path}')
print()
print('🎉 Setup complete. Proceed to 01_corpus_audit.ipynb')
